# FT-CUR com seleção aleatória — Tier 1, Tier 2 e Ablação D (2 T4, ≈2h30)

Última peça da troca `colnorm` → `random` decidida em 2026-09-19. A troca atinge os quatro modelos
que usam *landmarks*, e três já estão completos nos três regimes com 30 sementes: o Nyström-SVM (braço
de julho) e o ADMM-Nyström e o FISTA-Nyström (rodados em CPU local em 2026-10-05). Falta o **FT-CUR** nos
três regimes. O Tier 1 dele existe em `results/ftcur_sel_tier1.json`, mas foi gerado em 13/07/2026 —
**antes da correção da semeadura** (`d15f90d`, 19/09), que fez o FT-CUR e o SAINT deixarem de ser não
determinísticos. Usá-lo misturaria, dentro da mesma tabela, um Tier 1 sem semeadura com um Tier 2 e uma
Ablação D com semeadura; por 1 h de execução, refaz-se também o Tier 1 e o braço fica uniforme.

**Por que a troca.** O seletor `colnorm` amostra ∝ ‖x_i‖² no espaço de entrada, e sob o kernel RBF isso é
quase perfeitamente ANTICORRELACIONADO com a norma de coluna que o nome promete (Spearman −1,00 no σ que
o GridCV seleciona). A justificativa a priori da escolha caiu, e a opção adotada passa a ser a amostragem
aleatória uniforme — o baseline canônico, o mais barato, e o que não exige defesa. Não custa desempenho:
no FT-CUR o contraste é Δ = −0,0012 com p = 0,83 no ponto de operação.

**O `colnorm` não é refeito.** Ele continua nos arquivos onde está, usado em dois lugares apenas: como
ponto de comparação da Seção 5.7 (que compara orçamento com o seletor parado) e como um dos quatro
braços do Apêndice C, que é a própria ablação de seletores.

**Antes de rodar:** Settings → Accelerator → **GPU T4 x2**. Rode as células na ordem. A Ablação D depende
das modas que a célula do Tier 2 extrai, então não pule.


In [ ]:
# ── 1. GPUs ──
import torch
print('CUDA:', torch.cuda.is_available(), '| placas:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(' ', i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 2, 'Settings → Accelerator → GPU T4 x2 (com 1 placa, troque run_phase_2gpu por run_phase)'

In [ ]:
# ── 2. Repositório + verificação da instrumentação de orçamento ──
import os, subprocess, json, statistics as st
GIT_URL     = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
PROJECT_DIR = '/kaggle/working/sparse-lssvm-transformers-study'
BRANCH      = 'revisao/estatistica-e-proveniencia'
if os.path.exists(PROJECT_DIR):
    subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, GIT_URL, PROJECT_DIR], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase', 'origin', BRANCH], check=True)
os.chdir(PROJECT_DIR)
!git log --oneline -2
import sys; sys.path.insert(0, '.')

# A instrumentação tem de existir nos DOIS laços: o FT tem laço próprio, enquanto
# SAINT e FT-CUR passam por fit_model/train_epoch em ft_transformer_model.py.
import inspect
from src.models.transformers.ft_transformer import FTTransformer
assert 'self.best_epoch_' in inspect.getsource(FTTransformer.fit), 'FT sem best_epoch_ — clone velho, refaça o pull'
assert "'best_epoch'" in open('src/models/ft_transformer_model.py').read(), 'fit_model sem best_epoch'

import scripts.run_tier1_gridcv as _r1
assert hasattr(_r1, 'BUDGET_OVERRIDE') and hasattr(_r1, '_apply_budget'), 'runner sem override de orçamento'
# O override precisa acertar a chave de cada wrapper (FT usa max_epochs; SAINT/FT-CUR usam epochs).
_r1.BUDGET_OVERRIDE.update({'epochs': 200, 'patience': 16})
assert _r1._apply_budget({'max_epochs': 40, 'patience': 6}) == {'max_epochs': 200, 'patience': 16}
assert _r1._apply_budget({'epochs': 40, 'patience': 6})     == {'epochs': 200, 'patience': 16}
_r1.BUDGET_OVERRIDE.clear()
print('instrumentação OK: best_epoch/n_steps nos dois laços + override nas duas convenções de chave')

In [ ]:
# ── 3. Dependências e dados (tudo em cache no clone; nada é gerado nem baixado) ──
!pip install -q einops scikit-posthocs openpyxl 2>&1 | tail -n 1
from pathlib import Path
USADOS = ['BCW','PID','HAB','VCP','GCR','AUS','AI4I','TWS','TWM','TWC',        # Tier 1
          'ADULT','BANK','CREDIT','HIGGS50K','SHOPPERS','TELCO']               # Tier 2
falt = [d for d in USADOS if not Path(f'data/raw/{d.upper()}.parquet').exists()]
assert not falt, f'sem cache (seriam gerados em paralelo): {falt} — refaça o pull'
from src.data.loaders import DatasetLoader
for ds in USADOS:
    X, y, _ = DatasetLoader.load(ds)
print(f'{len(USADOS)} datasets OK — todos em cache, nenhuma escrita concorrente possível')

In [ ]:
# ── 4. Configuração, progresso e paralelismo nas duas placas ──
import json, os, shutil, time, subprocess, collections, statistics as st
from pathlib import Path

MODELS = ['FTTransformer_softmax', 'FTTransformer_topk', 'FTTransformer_entmax',
          'FTTransformer_sparsemax', 'SAINTColnorm', 'FTTransformerCURColnorm']
MODELS_STR = ' '.join(MODELS)
TIER1 = 'BCW PID HAB VCP GCR AUS AI4I TWS TWM TWC'
TIER2 = 'ADULT BANK CREDIT HIGGS50K SHOPPERS TELCO'
# Calibração do Tier 2: os três que mais custam e mais variam em passos por época.
TIER2_CALIB = 'BANK HIGGS50K TELCO'

# Orçamento do braço novo. min_epochs fica DESLIGADO de propósito: forçar um piso
# destruiria a medição de onde a paciência dispara — foi o que inviabilizou o piloto
# de 18/09, que usava min_epochs=200 e por isso não serve para estimar custo.
BUDGET = '--budget-epochs 200 --budget-patience 16'

SEEDS_CALIB = range(2)      # 1 semente por placa
SEEDS_FULL  = range(30)     # 15 por placa. 30 e nao 10: a Secao 5.7 passou a ser
                            # carregadora de peso (o SAINT sobe 12 posicoes no Tier 1),
                            # e com 10 sementes a afirmacao central teria um terco da
                            # amostra das tabelas que ela contradiz. Custo medido na
                            # calibracao: Tier 1 9,4 h e Tier 2 ~11,9 h nas duas placas.

OUT = {'c1pub': 'results/budget_calib_t1_pub.json',  'c1new': 'results/budget_calib_t1_200.json',
       'c2pub': 'results/budget_calib_t2_pub.json',  'c2new': 'results/budget_calib_t2_200.json',
       'f1new': 'results/budget_full_t1_200.json',   'f2new': 'results/budget_full_t2_200.json'}
nm, nc, nf = len(MODELS), len(list(SEEDS_CALIB)), len(list(SEEDS_FULL))
TOTAL = {'c1pub': nm*10*nc, 'c1new': nm*10*nc,
         'c2pub': nm*3*nc,  'c2new': nm*3*nc,
         'f1new': nm*10*nf, 'f2new': nm*6*nf}
Path('results').mkdir(exist_ok=True)

RESUME_DIR = None    # ex.: Path('/kaggle/input/budget-parcial') para retomar sessão anterior
if RESUME_DIR:
    # Restaura o arquivo mesclado E os SHARDS por placa. Os shards são o que
    # importa: no modo de duas placas cada processo grava e retoma do seu próprio
    # --output (<base>_g0.json / _g1.json), então restaurar apenas o nome mesclado
    # faria os dois recomeçarem do zero. (Corrigido em 2026-10-05, depois de a
    # sessão do Tier 2 estourar o tempo.)
    _rest = 0
    for k, v in OUT.items():
        for nome in (Path(v).name,
                     Path(v).name.replace('.json', '_g0.json'),
                     Path(v).name.replace('.json', '_g1.json')):
            src = Path(RESUME_DIR) / nome
            if src.exists():
                dst = Path('results') / nome
                shutil.copy(src, dst)
                try:
                    _n = len([r for r in json.load(open(dst)) if r.get('status') == 'ok'])
                except Exception:
                    _n = '?'
                print(f'restaurado {dst}  ({_n} execuções ok)')
                _rest += 1
    print(f'{_rest} arquivo(s) restaurado(s) — os runners pulam o que já está feito')

def save(key):
    shutil.copy(OUT[key], '/kaggle/working/' + Path(OUT[key]).name)
    print('salvo em Output:', Path(OUT[key]).name)

def _mirror(*paths):
    # Os runners gravam em <repo>/results/ a cada execução (append atômico), mas o painel
    # Output do Kaggle mostra /kaggle/working: sem esta cópia periódica, uma sessão
    # interrompida deixaria os dados só no diretório aninhado do clone.
    for p in paths:
        try:
            if Path(p).exists(): shutil.copy(p, '/kaggle/working/' + Path(p).name)
        except Exception as e:
            print('  (aviso: falha ao espelhar', p, e, ')')

def _count(path):
    # Só erros ESPERADOS são tolerados (arquivo ainda não criado / meio de escrita).
    # Um except amplo aqui esconderia um bug de import e o contador ficaria em 0 para
    # sempre com os processos rodando — foi o que aconteceu em 2026-09-19 no SAINT.
    try: r = json.load(open(path))
    except (FileNotFoundError, ValueError): return 0, ''
    ok = [x for x in r if x.get('status', 'ok') == 'ok']
    if not ok: return 0, ''
    x = ok[-1]; f1 = x.get('test_f1_macro')
    info = 'último: ' + str(x.get('dataset', '?')) + '/seed' + str(x.get('seed', '?'))
    if isinstance(f1, (int, float)): info += f' F1={f1:.3f}'
    if x.get('best_epoch') is not None:
        info += f" best_ep={x['best_epoch']}/{x.get('n_epochs','?')}"
    return len(ok), info

def run_phase_2gpu(key, cmd_fmt, seeds, every=60, heartbeat=900):
    # Duas T4: um processo por placa, metade das sementes cada, shards separados.
    # NÃO usar DataParallel/DDP: FT-CUR e SAINT fazem atenção inter-instâncias DENTRO
    # do lote, então dividir o lote entre placas mudaria o modelo. O paralelismo aqui é
    # por semente, que não altera nada. Dois processos no mesmo JSON se sobrescreveriam,
    # daí um arquivo por placa.
    seeds = list(seeds); half = max(1, len(seeds)//2)
    base, total = OUT[key], TOTAL[key]
    shards, procs, logs = [], [], []
    for gpu, sds in [(0, seeds[:half]), (1, seeds[half:])]:
        if not sds: continue
        shard = base.replace('.json', f'_g{gpu}.json'); shards.append(shard)
        log = f'/kaggle/working/{key}_g{gpu}.log'; logs.append(log)
        cmd = cmd_fmt.format(seeds=' '.join(map(str, sds)), output=shard)
        lf = open(log, 'w')
        # CUDA_VISIBLE_DEVICES prende o processo a uma placa; os limites de thread evitam
        # que os dois disputem os poucos vCPUs do Kaggle (oversubscription de BLAS/OMP
        # chega a dobrar o tempo do pré-processamento).
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu),
                   OMP_NUM_THREADS='2', MKL_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2',
                   NUMEXPR_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false')
        procs.append(subprocess.Popen(cmd, shell=True, stdout=lf, stderr=subprocess.STDOUT, env=env))
        print(f'[{key}] GPU {gpu}: sementes {sds[0]}..{sds[-1]} -> {Path(shard).name}', flush=True)
    t0 = time.time(); last_n, last_print = -1, 0.0
    while True:
        rcs = [p.poll() for p in procs]
        n = sum(_count(s)[0] for s in shards); now = time.time()
        if n != last_n or all(rc is not None for rc in rcs) or now - last_print > heartbeat:
            el = (now - t0)/60; eta = el/n*(total-n) if n else float('nan')
            _, info = _count(shards[0])
            print(f'[{key}] {n:>4}/{total}  {100*n/total:3.0f}%  {el:6.0f} min  ETA {eta:5.0f} min  {info}', flush=True)
            _mirror(*shards)
            last_n, last_print = n, now
        if all(rc is not None for rc in rcs): break
        time.sleep(every)
    recs = []
    for s in shards:
        try: recs += json.load(open(s))
        except Exception: pass
    Path(base).write_text(json.dumps(recs, indent=1))
    print(f'[{key}] terminou (exits={rcs}) — {len(recs)} registros'); save(key)
    for s in shards: shutil.copy(s, '/kaggle/working/' + Path(s).name)
    if any(rc != 0 for rc in rcs): raise RuntimeError(f'{key}: exits {rcs} — ver {logs}')

for _n in ('json', 'os', 'shutil', 'time', 'subprocess', 'collections', 'st', 'Path'):
    assert _n in dir() or _n in globals(), f'falta import: {_n}'
print('config OK —', len(MODELS), 'Transformers | orçamento novo:', BUDGET)
for k, v in OUT.items(): print(f'  {k:<6} -> {v}  (total {TOTAL[k]})')

In [ ]:
# ── 5. Tier 1 (10 datasets × 30 sementes) ──
# Refeito apesar de existir em results/ftcur_sel_tier1.json: aquele arquivo é de 13/07/2026,
# anterior à correção da semeadura do torch (19/09). Sem isto, o braço random do FT-CUR teria
# Tier 1 não determinístico e Tier 2 determinístico na mesma tabela.
MODELS = ['FTTransformerCURRandom']
MODELS_STR = ' '.join(MODELS)
TIER1 = 'BCW PID HAB VCP GCR AUS AI4I TWS TWM TWC'
TIER2 = 'ADULT BANK CREDIT HIGGS50K SHOPPERS TELCO'
SEEDS = range(30)
OUT['t1'], TOTAL['t1'] = 'results/tier1_ftcur_random.json', 1 * 10 * 30
OUT['t2'], TOTAL['t2'] = 'results/tier2_ftcur_random.json', 1 * 6 * 30
OUT['ablD'], TOTAL['ablD'] = 'results/tier2_fixedparams_n5000_ftcur_random.json', 1 * 6 * 30
print('alvos:', {k: OUT[k] for k in ('t1', 't2', 'ablD')})

run_phase_2gpu('t1', "python -u scripts/run_tier1_gridcv.py --models " + MODELS_STR +
               " --datasets " + TIER1 +
               " --seeds {seeds} --output {output} --log-level WARNING",
               seeds=SEEDS)


In [ ]:
# ── 6. Tier 2 (N=2000, 6 datasets × 30 sementes) ──
run_phase_2gpu('t2', "python -u scripts/run_tier2_gridcv.py --models " + MODELS_STR +
               " --datasets " + TIER2 + " --n-train 2000"
               " --seeds {seeds} --output {output} --log-level WARNING",
               seeds=SEEDS)


In [ ]:
# ── 7. Ablação D (N=5000, hiperparâmetros FIXOS = moda do Tier 2 recém-rodado) ──
# Mesmo procedimento da Ablação D original: a moda das configurações vencedoras do
# GridSearchCV em N=2000, por dataset. A chave FTTransformerCURRandom ainda não existe
# no config, por isso é criada aqui a partir do que a célula 5 produziu.
recs = [r for r in json.load(open(OUT['t2'])) if r.get('status') == 'ok' and r.get('best_params')]
assert len(recs) == TOTAL['t2'], f'esperava {TOTAL["t2"]} registros ok, achei {len(recs)}'
cfg = json.load(open('config/tier2_fixed_params.json'))
by = collections.defaultdict(list)
for r in recs:
    by[(r['variant'], r['dataset'])].append(tuple(sorted(r['best_params'].items())))
for (v, d), vals in sorted(by.items()):
    mode, cnt = collections.Counter(vals).most_common(1)[0]
    cfg.setdefault(v, {})[d] = dict(mode)
    print(f'{v:24s} {d:10s} {dict(mode)}  [moda {cnt}/{len(vals)}]')
Path('config/tier2_fixed_params_ftcur_random.json').write_text(json.dumps(cfg, indent=2, sort_keys=True))
shutil.copy('config/tier2_fixed_params_ftcur_random.json', '/kaggle/working/')

run_phase_2gpu('ablD', "python -u scripts/run_tier2_fixedparams.py --models " + MODELS_STR +
               " --datasets " + TIER2 + " --n-train 5000"
               " --config config/tier2_fixed_params_ftcur_random.json"
               " --seeds {seeds} --output {output} --log-level WARNING",
               seeds=SEEDS)


In [ ]:
# ── 8. Conferência e comparação com o braço colnorm ──
import statistics as st
for key, ref, rv in (('t1', 'results/tier1_gridcv.json', 'FTTransformerCURColnorm'),
                     ('t2', 'results/tier2_transformers.json', 'FTTransformerCURColnorm'),
                     ('ablD', 'results/tier2_fixedparams_n5000_transformers.json', 'FTTransformerCURColnorm')):
    d = json.load(open(OUT[key])); ok = [r for r in d if r.get('status') == 'ok']
    print(f"\n=== {key}: {len(ok)}/{TOTAL[key]} ok ===")
    a = {(r['dataset'], r['seed']): r['test_f1_macro']
         for r in json.load(open(ref)) if r.get('variant') == rv and r.get('status') == 'ok'}
    b = {(r['dataset'], r['seed']): r['test_f1_macro'] for r in ok}
    ch = sorted(set(a) & set(b))
    if ch:
        dd = [b[k] - a[k] for k in ch]
        print(f"  colnorm {st.mean([a[k] for k in ch]):.4f}  ->  random {st.mean([b[k] for k in ch]):.4f}"
              f"   Δ={st.mean(dd):+.4f}  ({sum(1 for x in dd if x > 0)}/{len(dd)} vitórias)")
    for ds in sorted(set(k[0] for k in b)):
        v = [x for (d_, s), x in b.items() if d_ == ds]
        print(f"  {ds:10s} {st.mean(v):.4f} ± {st.pstdev(v):.3f}  n={len(v)}")
print('\nBaixe de /kaggle/working: tier1_ftcur_random.json, tier2_ftcur_random.json, '
      'tier2_fixedparams_n5000_ftcur_random.json e tier2_fixed_params_ftcur_random.json')
